# Fontes de imagens pré-treinadas

* Roboflow Universe
  * https://universe.roboflow.com/browse/manufacturing
* Kaggle
* Google Open Images v7 dataset

## Datasets selecionados

No Roboflow Universe foi possível encontrar modelos de fotos de caixas empilhadas. Foi adicinado em um novo projeto e dataset v1:

* https://app.roboflow.com/embrapac/products-counter/1

In [1]:
# checar se GPU disponível
!nvidia-smi

Sat Mar 21 14:53:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Instalando dependências

In [2]:
#!pip uninstall -y numpy ultralytics roboflow
#!pip install roboflow==1.1.48 ultralytics==8.2.103 numpy==1.26.4
!pip install roboflow==1.1.48 ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


# Versões das bibliotecas e hardware local

In [3]:
from IPython import display
display.clear_output()

# prevent ultralytics from tracking your activity
!yolo settings sync=False

import ultralytics
ultralytics.checks()

Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.6/112.6 GB disk)


# Obtenção do dataset criado e rotulado no RoboFlow

In [4]:
from roboflow import Roboflow
from google.colab import userdata

# Novas versões de dataset podem ser criados nos projetos. Especifique a versão do dataset:
dataset_version = 1
dataset_name=f"exp_small_dataset_v{dataset_version}"
# Dados do projeto RoboFlow utilizado
ROBOFLOW_WORKSPACE = "embrapac"
ROBOFLOW_PROJECT = "products-counter"
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(dataset_version)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to products-counter-1 in yolov8:: 100%|██████████| 19702/19702 [00:04<00:00, 4848.90it/s]


In [5]:
# from google.colab import drive
# drive.mount('/content/drive')

# Treinamento do modelo

In [ ]:
from ultralytics import YOLO

folder = '{}-{}'.format(ROBOFLOW_PROJECT, dataset_version)

# caminho para o data.yaml baixado pelo Roboflow no filesystem do Colab
DATA_YAML = f"/content/{folder}/data.yaml"
# caminho para o data.yaml quando roda local
# DATA_YAML = "data.yaml"

# escolhe o modelo base voltado para execução num RPi 5
# model = YOLO("yolov8n.pt")
# model = YOLO("yolov8s.pt")
# o modelo médio entrega mAP significativamente maior a um custo computacional moderado.​
# O yolov8m tem maior capacidade de feature extraction justamente para cenários Com fotos tiradas de tempos em tempos de caixas acumuladas, até 2-3 segundos de latência por inferência são aceitáveis.
# Além disso, pode aumentar o imgsz na inferência sem custo de latência crítico
model = YOLO("yolov8m.pt")

# treinamento teste 1 = NOK
# model.train(
#     data=DATA_YAML,
#     epochs=35,
#     patience=7,          # Aguarda 50 epochs sem melhora antes de parar (early stopping)
#     imgsz=640,
#     batch=16,
#     # --- ORGANIZAÇÃO DOS EXPERIMENTOS ---
#     project="embrapac_runs",
#     name=dataset_name,
#     plots=True,           # Salva gráficos de loss, mAP, etc.
# )
# treinamento teste 2 =
model.train(
    data=DATA_YAML,
    epochs=150,
    patience=30,
    imgsz=1280,    # Resolução maior — detecta caixas menores/oclusas melhor
    batch=8,              # Reduzido de 16 → 8 por causa do imgsz maior (VRAM)
    optimizer="SGD",      # Melhor para datasets grandes (~10k+)
    lr0=0.01,             # LR padrão para SGD (era 0.001 do AdamW — importante mudar!)
    lrf=0.01,             # Learning rate final = lr0 * lrf
    momentum=0.937,       # Padrão Ultralytics para SGD
    weight_decay=0.0005,  # Regularização L2 contra overfitting
    cos_lr=True,          # Scheduler coseno: melhor decaimento da LR
    warmup_epochs=5,      # Estabiliza o início do treinamento
    close_mosaic=15,      # Desabilita mosaic nos últimos 15 epochs para estabilizar
    mixup=0.1,            # Blending entre imagens, melhora generalização
    copy_paste=0.2,       # Aumentado: ajuda com oclusão entre caixas acumuladas
    degrees=10.0,         # Rotação suave
    scale=0.6,            # Variação de escala (detectar caixas em distâncias variadas)
    project="embrapac_runs",
    name=dataset_name,
    plots=True,
    cache=False,          # Cache=False com imgsz=1280 para evitar OOM no Colab
)



Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/products-counter-1/data.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp_small_dataset_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, pa

In [ ]:
!zip -r resultado_treinamento.zip /content/runs/detect/embrapac_runs/

from google.colab import files
files.download('resultado_treinamento.zip')


# Teste do modelo

## Testando localmente em Ubuntu 24.04 com AMD Radeon

A partir do arquivo de saída do modelo já treinado (`/content/runs/detect/train/weight/best.pt`) parece possível rodar a inferência num PC com placa AMD na expectativa de conseguir testar detecções em imagens/vídeos a ~5-15 FPS dependendo da resolução.

A identificação da GPU do PC local foi obtido da seguinte forma:

```
$ lspci | grep -i vga
06:00.0 VGA compatible controller: Advanced Micro Devices, Inc. [AMD/ATI] Renoir [Radeon RX Vega 6 (Ryzen 4000/5000 Mobile Series)] (rev d3)
```

### Criando ambiente Python local

Virtual Environment criado para python 3.12.3: `pyenv/yolo`

Instalar as seguintes dependências para testar o modelo localmente:

`pip install ultralytics opencv-python`

Depois, para inferência (testar em imagens ou vídeo), você usa esse best.pt:

```
from ultralytics import YOLO

model = YOLO("best.pt")  # caminho do best.pt que você treinou

# testar em uma imagem
results = model("alguma_imagem.jpg", save=True)

# ou em vídeo
results = model("video.mp4", save=True)
```

## Testando por foto no Colab (próxima seção)

In [ ]:
# CÓDIGO PARA TESTAR UM MODELO TREINADO LOCALMENTE, O QUE CONSIDERA O best.pt CARREGADO MANUALMENTE AQUI

from ultralytics import YOLO

trained_model = f"/content/runs/detect/embrapac_runs/{dataset_name}/weights/best.pt"

model = YOLO(trained_model)

# classes do dataset
print('-------------------- CLASSES DO MODELO')
print(model.names)
print('--------------------')

# Sugestão pelo V3 a partir da análise dos gráficos Box:
# Para começar a inferência no Raspberry Pi 5: confidence entre 0,25 e 0,35
# Prioridade for evitar disparos falsos no background, subir para algo como 0,40–0,50

EXPECTED_CONF = 0.5
# results = model("/content/caixa_com_etiqueta3.jpg", save=True)  # save=True salva as imagens anotadas em runs/detect/predict/

#results = model.predict(source="foto.jpg", conf=0.25, iou=0.45)

results = model("/content/IMG_20260309_100329.jpg", conf=EXPECTED_CONF)

print(results)
if results is None:
  print("Nenhum resultado encontrado")
  exit()

# results é uma LISTA de Results (1 item se for 1 imagem)
print(f"Processou {len(results)} frames/imagens")

# Pega o primeiro resultado (results[0])
r = results[0]
# 1. Ver a imagem anotada (abre janela)
r.show()

# 2. Extrair as detecções manualmente (para lógica customizada)
if r.boxes is not None:  # se achou objetos
    print(f"Achou {len(r.boxes)} objetos")

    for box in r.boxes:
        # Coordenadas da bbox: [x1, y1, x2, y2] em pixels
        xyxy = box.xyxy[0].tolist()
        print(f"Bbox: {xyxy}")

        # Classe (0=classe1, 1=classe2, etc.) - confere no seu data.yaml
        cls_id = int(box.cls[0])
        print(f"Classe ID: {cls_id}")
        # Confiança (0-1)
        conf = float(box.conf[0])
        print(f"Confiança: {conf:.2f}")

        if conf > EXPECTED_CONF:  # threshold típico
            print("→ OBJETO VÁLIDO DETECTADO")
